# Preprocesamiento, Análisis Exploratorio y Entrenamiento de Modelos

### Estructura del notebook

| Sección | Contenido |
|---|---|
| 0–2 | Instalación, importaciones y parámetros globales |
| 3–6 | Carga, corrección de año, fusión LB × V-Dem, exclusiones |
| 7–8 | Recodificación del target y transformaciones de escala |
| **EDA Parte 1** | Exploración sobre datos transformados pre-split |
| 9–10 | Definición de features y construcción de splits temporales |
| **EDA Parte 2** | Validación post-transformación y selección empírica de features |
| 11–12 | Imputación diferenciada y función de evaluación |
| 13–17 | Cinco modelos con optimización Optuna |
| 18 | Ciclo principal de entrenamiento y guardado del pipeline completo |
| 19–21 | Función de predicción en producción, resultados y visualización |
| 22 | Registro de versiones |

> **Regla de oro anti-data leakage:** toda transformación aprendida (imputador, scaler)
> se ajusta **exclusivamente** sobre el conjunto de entrenamiento de cada split.

## 1. Importaciones

In [ ]:
# =============================================================================
# Importaciones globales
# =============================================================================
import os, gc, json, joblib, warnings, logging
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import missingno as msno

from sklearn.experimental import enable_iterative_imputer   # noqa
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import BayesianRidge, LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    balanced_accuracy_score, f1_score, cohen_kappa_score,
    roc_auc_score, mean_absolute_error,
)

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

from tqdm.notebook import tqdm

# Agregar raiz del proyecto al path (funciona en VS Code y Jupyter Lab)
import sys

_root = Path(".").resolve()
if not (_root / "utils").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

print("✓ Importaciones completadas.")

Configuraciones personalizadas

In [ ]:
from utils.config import setup_plots
from utils.config import THEME
from utils.config import PALETTES
from utils.config import SEED

from utils.plots import save_figure
from utils.plots import model_color

setup_plots()

## 2. Parámetros globales del pipeline

> Toda configuración ajustable se concentra aquí.
> Modificar este bloque sin tocar el resto del notebook.

In [ ]:
# =============================================================================
# PARÁMETROS GLOBALES — modificar aquí para ajustar cualquier configuración
# =============================================================================

# ── Reproducibilidad ──────────────────────────────────────────────────────────
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Hardware ───────────────────────────────────────────────────────────────────
USAR_GPU       = True
DISPOSITIVO_TN = "cuda" if USAR_GPU and torch.cuda.is_available() else "cpu"
N_JOBS         = -1

# ── Rutas ─────────────────────────────────────────────────────────────────────
RAIZ           = Path("..")
CARPETA_DATA   = RAIZ / "data"
CARPETA_BASE   = CARPETA_DATA / "base"
CARPETA_PROC   = CARPETA_DATA / "processed"
CARPETA_MODELS = RAIZ / "models"
CARPETA_RESULTS= RAIZ / "results"
for d in [CARPETA_PROC, CARPETA_MODELS, CARPETA_RESULTS, CARPETA_RESULTS / "figures", CARPETA_RESULTS / "metrics", CARPETA_RESULTS / "tables"]:
    d.mkdir(parents=True, exist_ok=True)

ARCHIVO_LB   = CARPETA_BASE / "latinobarometro.csv"
ARCHIVO_VDEM = CARPETA_BASE / "v-dem.csv"

# ── Flags de ejecución ────────────────────────────────────────────────────────
EJECUTAR_BUSQUEDA_HP = True   # False carga HPs ya guardados si existen
N_TRIALS_OPTUNA      = 50     # Reducir a 20 para pruebas rápidas

# ── Subperíodos — Expanding Window Walk-Forward ───────────────────────────────
SUBPERIODOS = {
    "SP1": {
        "train_olas": [1995,1996,1997,1998,2000,2001,2002,2003,2004,2005],
        "validate_ola"  : [2006],
        "test_ola"  : [2007],
        "descripcion": "Consolidación democrática (1995–2007)",
    },
    "SP2": {
        "train_olas": [1995,1996,1997,1998,2000,2001,2002,2003,2004,2005,2006,
                       2007,2008,2009,2010,2011,2013,2015,2016],
        "validate_ola"  : [2017],
        "test_ola"  : [2018],
        "descripcion": "Crisis y reconfiguración (2008–2018)",
    },
    "SP3": {
        "train_olas": [1995,1996,1997,1998,2000,2001,2002,2003,2004,2005,2006,
                       2007,2008,2009,2010,2011,2013,2015,2016,2017,2018,2020],
        "validate_ola"  : [2023],
        "test_ola"  : [2024],
        "descripcion": "Pandemia y recuperación (2020–2024)",
    },
}
N_SPLITS_CV = 5

# ── Subregiones ───────────────────────────────────────────────────────────────
SUBREGIONES = {
    "Cono Sur"       : ["Argentina","Chile","Uruguay","Paraguay","Perú"],
    "Región Andina"  : ["Bolivia","Colombia","Ecuador","Venezuela"],
    "Brasil"         : ["Brasil"],
    "Centroamérica"  : ["Costa Rica","El Salvador","Guatemala",
                        "Honduras","Nicaragua","Panamá"],
    "México y Caribe": ["México","República Dominicana"],
}

# ── Tratamiento especial de países ────────────────────────────────────────────
PAISES_EXCLUIR_TEST = ["Venezuela", "Nicaragua"]
AÑO_CORTE_VEN       = 2017

# ── Mapeos de identificación ──────────────────────────────────────────────────
MAPEO_NUMINVES = {16: 2011, 17: 2013, 18: 2015, 23: 2023, 24: 2024}

MAPEO_PAIS_ISO3 = {
    "Argentina": "ARG", "Bolivia": "BOL", "Brasil": "BRA",
    "Chile": "CHL", "Colombia": "COL", "Costa Rica": "CRI",
    "República Dominicana": "DOM", "Ecuador": "ECU",
    "El Salvador": "SLV", "Guatemala": "GTM", "Honduras": "HND",
    "México": "MEX", "Nicaragua": "NIC", "Panamá": "PAN",
    "Paraguay": "PRY", "Perú": "PER", "Uruguay": "URY",
    "Venezuela": "VEN",
}

# ── Nombres de columnas clave ─────────────────────────────────────────────────
COL_TARGET = "target"
COL_AÑO    = "año"
COL_PAIS   = "pais_nombre"
COL_ISO3   = "pais_iso3"
COL_PESO   = "X_020"

# ── Códigos NS/NR → NaN ───────────────────────────────────────────────────────
NSNR = [-1, -2, -3, -4, -5, -6, -7, -8]

# ── Variables excluidas ───────────────────────────────────────────────────────
VARS_EXCLUIR_LB = [
    "C_001_031",  # ruptura de codificación en 2018; sin señal en ningún subperiodo
    "A_003_021",  # ausente en SP2_test y SP3_test; distorsionaría SHAP
    "D_001_061",  # ausente en los 3 conjuntos de prueba
    "D_001_131",  # ausente en SP2_test y SP3_test
    "X_004",      # 627 categorías; 94% nuevas en SP3_test; sin señal predictiva
    "S_700",      # sin señal en ningún subperiodo ni subregión; alta cardinalidad
]

VARS_EXCLUIR_VDEM = [
    "v2x_neopat",     # correlación 0.970 con v2x_rule — redundancia crítica
    "v2xnp_regcorr",  # correlación 0.985 con v2x_execorr — redundancia crítica
    "v2xpe_exlsocgr", # sin datos en todos los países en 2024 (SP3_test)
    "v2xpe_exlecon",  # sin datos en todos los países en 2024 (SP3_test)
]

# ── Transformaciones de escala ────────────────────────────────────────────────
# Confianza institucional: escala 1=Mucho → 4=Nada → invertir (5 - x)
VARS_LIKERT4_INVERTIR = [
    "H_002_011", "H_002_031", "H_002_041", "H_002_101",
    "H_002_111", "H_002_131", "H_002_161", "H_002_241",
    "G_005_001",
]
# Interés en política: escala 1=Mucho → 4=Ninguno → invertir (5 - x)
VARS_LIKERT4_INTERES = ["A_007_001"]

# V-Dem con dirección invertida: mayor = más corrupción → aplicar (1 - x)
VARS_VDEM_INVERTIR = ["v2x_corr", "v2x_execorr", "v2x_pubcorr"]

# Variables categóricas nominales
# S_700 excluida por ausencia de señal en todos los subperiodos y subregiones.
# S_002 (edad) se mantiene: justificación teórica de efectos de cohorte.
# X_008: tamaño de municipio (1=rural → 8=metropolitano); sin inversión necesaria.
VARS_CATEGORICAS = ["S_200"]

# ── Número de clases del target ───────────────────────────────────────────────
N_CLASES = 4

# ── Etiquetas del target ──────────────────────────────────────────────────────
ETIQUETAS = {
    0: "Para nada satisfecho",
    1: "No muy satisfecho",
    2: "Más bien satisfecho",
    3: "Muy satisfecho",
}

# ── Bloques temáticos y etiquetas de features ─────────────────────────────────
# Organización en seis bloques para documentación y visualizaciones.
# Los bloques agrupan variables según su dimensión conceptual en la literatura
# de ciencias políticas (Easton 1975, Norris 2011, Lewis-Beck & Stegmaier 2000).

BLOQUES = {
    "Confianza institucional": [
        "H_002_011", "H_002_031", "H_002_041", "H_002_101",
        "H_002_111", "H_002_131", "H_002_161", "H_002_241",
        "H_001_011",
    ],
    "Evaluación económica": [
        "D_001_001", "D_001_021", "D_001_041", "D_001_091",
        "C_003_003_011", "C_006_003_011",
    ],
    "Percepción política": [
        "A_001_001", "A_007_071", "A_007_001",
        "B_001_101", "B_006_061",
    ],
    "Corrupción y seguridad": [
        "G_002_011", "G_005_001", "I_001_001",
    ],
    "Características sociodemográficas": [
        "S_001", "S_002", "S_101", "S_200", "S_301", "S_701", "X_008",
    ],
    "Contexto democrático · High-level": [
        "v2x_polyarchy", "v2x_libdem", "v2x_partipdem",
        "v2x_delibdem", "v2x_egaldem",
    ],
    "Contexto democrático · Mid-level": [
        "v2xcl_rol", "v2x_jucon", "v2xlg_legcon", "v2x_freexp_altinf",
        "v2x_cspart", "v2xcs_ccsi",
        "v2x_egal", "v2xeg_eqdr",
        "v2x_accountability_osp", "v2x_rule",
        "v2x_corr", "v2x_execorr", "v2x_pubcorr",
        "v2x_gender", "v2x_polyarchy",
    ],
}

# Diccionario plano: variable → etiqueta corta para gráficos
ETIQUETAS_FEATURES = {
    # ── Confianza institucional ────────────────────────────────────────────────
    "H_002_011": "Confianza Congreso",
    "H_002_031": "Confianza Gobierno",
    "H_002_041": "Confianza Poder Judicial",
    "H_002_101": "Confianza Iglesia Católica",
    "H_002_111": "Confianza Policía",
    "H_002_131": "Confianza Televisión",
    "H_002_161": "Confianza FF.AA.",
    "H_002_241": "Confianza Partidos Políticos",
    "H_001_011": "Confianza interpersonal",
    
    # ── Evaluación económica ──────────────────────────────────────────────────
    "D_001_001": "Situación económica país",
    "D_001_021": "Economía país vs. año anterior",
    "D_001_041": "Expectativa económica país",
    "D_001_091": "Expectativa económica personal",
    "C_003_003_011": "Preocupación desempleo",
    "C_006_003_011": "Distribución ingreso justa",
    
    # ── Percepción política ───────────────────────────────────────────────────
    "A_001_001": "Apoyo a la democracia",
    "A_007_071": "Escala Izquierda-Derecha",
    "A_007_001": "Interés en política",
    "B_001_101": "País para todos / poderosos",
    "B_006_061": "Aprobación gobierno",
    
    # ── Corrupción y seguridad ────────────────────────────────────────────────
    "G_002_011": "Conoce caso de corrupción",
    "G_005_001": "Progreso contra corrupción",
    "I_001_001": "Victimización delictiva",
    
    # ── Características sociodemográficas ─────────────────────────────────────
    "S_001": "Sexo",
    "S_002": "Edad",
    "S_101": "Nivel educativo",
    "S_200": "Situación ocupacional",
    "S_301": "Nivel socioeconómico",
    "S_701": "Práctica religiosa",
    "X_008": "Tamaño municipio (urbano-rural)",
    
    # ── Contexto democrático · High-level ────────────────────────────────────
    "v2x_polyarchy": "Democracia electoral",
    "v2x_libdem"   : "Democracia liberal",
    "v2x_partipdem": "Democracia participativa",
    "v2x_delibdem" : "Democracia deliberativa",
    "v2x_egaldem"  : "Democracia igualitaria",
    
    # ── Contexto democrático · Mid-level ─────────────────────────────────────
    "v2xcl_rol"           : "Igualdad ante la ley",
    "v2x_jucon"           : "Independencia judicial",
    "v2xlg_legcon"        : "Control legislativo",
    "v2x_freexp_altinf"   : "Libertad de expresión",
    "v2x_cspart"          : "Participación soc. civil",
    "v2xcs_ccsi"          : "Índice soc. civil (CSI)",
    "v2x_egal"            : "Componente igualitario",
    "v2xeg_eqdr"          : "Distribución igualitaria",
    "v2x_accountability_osp": "Rendición de cuentas",
    "v2x_rule"            : "Estado de derecho",
    "v2x_corr"            : "Integridad institucional",
    "v2x_execorr"         : "Integridad ejecutiva",
    "v2x_pubcorr"         : "Integridad sector público",
    "v2x_gender"          : "Empoderamiento político mujeres",
}

# Helper: obtener bloque de una variable
def bloque_de(var: str) -> str:
    for bloque, variables in BLOQUES.items():
        if var in variables:
            return bloque
    return "Sin clasificar"

print("✓ Parámetros globales cargados.")
print(f"  GPU disponible : {torch.cuda.is_available()}")
print(f"  Dispositivo    : {DISPOSITIVO_TN}")
print(f"  Semilla        : {SEED}")
print(f"  Bloques temáticos definidos: {len(BLOQUES)}")
print(f"  Variables con etiqueta: {len(ETIQUETAS_FEATURES)}")

In [ ]:
torch.cuda.is_available() 

## 3. Carga de datos consolidados

In [ ]:
# =============================================================================
# Carga de Latinobarómetro y V-Dem
# =============================================================================
print("Cargando Latinobarómetro...")
df_lb_raw = pd.read_csv(ARCHIVO_LB, low_memory=False, encoding="utf-8-sig")
print(f"  {df_lb_raw.shape[0]:,} registros × {df_lb_raw.shape[1]} columnas")

print("Cargando V-Dem...")
df_vdem_raw = pd.read_csv(ARCHIVO_VDEM, low_memory=False, encoding="utf-8-sig")
print(f"  {df_vdem_raw.shape[0]:,} registros × {df_vdem_raw.shape[1]} columnas")

print()
print("Muestra LB (3 filas, primeras 8 columnas):")
print(df_lb_raw.iloc[:3, :8].to_string())
print()
print("Muestra V-Dem (3 filas):")
print(df_vdem_raw.head(3).to_string())

## 4. Corrección de la variable año y clave de fusión

`X_002` contiene el año calendario en la mayoría de olas, pero en cinco olas usa
el número de investigación: `16=2011`, `17=2013`, `18=2015`, `23=2023`, `24=2024`.
Esta corrección se aplica como primer paso antes de cualquier otra operación.

In [ ]:
# =============================================================================
# Corrección de X_002 → año calendario
# =============================================================================
df_lb = df_lb_raw.copy()

if "ola" in df_lb.columns:
    df_lb[COL_AÑO] = df_lb["ola"].str.replace("LAT", "", regex=False).astype(int)
    print("✓ Año extraído desde columna 'ola'")
else:
    df_lb[COL_AÑO] = df_lb["X_002"].map(
        lambda x: MAPEO_NUMINVES.get(int(x), int(x))
    )
    print("✓ Año derivado de X_002 con corrección de numinves")

print()
print("Distribución de registros por año:")
for año, n in df_lb[COL_AÑO].value_counts().sort_index().items():
    print(f"  {int(año)}: {n:,}")

# Derivar ISO3
if COL_ISO3 not in df_lb.columns:
    print(f"\nDerivando ISO3 desde {COL_PAIS}...")
    df_lb[COL_ISO3] = df_lb[COL_PAIS].map(MAPEO_PAIS_ISO3)

n_sin = df_lb[COL_ISO3].isna().sum()
if n_sin > 0:
    print(f"\n⚠ Países sin ISO3: {df_lb[df_lb[COL_ISO3].isna()][COL_PAIS].unique()}")
else:
    print(f"\n✓ ISO3 derivado sin errores ({df_lb[COL_ISO3].nunique()} países)")

del df_lb_raw
gc.collect()

Eliminar registros sin país

In [ ]:
# Eliminar todos los registros que no tengan nombre de país o año, ya que no se pueden usar para entrenamiento ni validación

df_lb = df_lb.dropna(subset=[COL_PAIS, COL_AÑO])


In [ ]:
print(f"\nTamaño de la muestra final: {df_lb.shape[0]:,} registros × {df_lb.shape[1]} columnas")

## 5. Fusión Latinobarómetro × V-Dem

In [ ]:
# =============================================================================
# Merge LB × V-Dem por clave (pais_iso3, año)
# =============================================================================
df_vdem = df_vdem_raw.copy().rename(
    columns={"country_text_id": COL_ISO3, "year": COL_AÑO}
)
cols_drop = ["country_name", "country_id", "COWcode"] + VARS_EXCLUIR_VDEM
df_vdem   = df_vdem[[c for c in df_vdem.columns if c not in cols_drop]]

n_vdem_feat = len([c for c in df_vdem.columns if c not in [COL_ISO3, COL_AÑO]])
print(f"Variables V-Dem a fusionar: {n_vdem_feat}")

df = df_lb.merge(df_vdem, on=[COL_ISO3, COL_AÑO], how="left", validate="m:1")

n_sin_vdem = df["v2x_polyarchy"].isna().sum()
pct_ok     = (1 - n_sin_vdem / len(df)) * 100
print(f"\nResultado del merge:")
print(f"  Registros totales  : {len(df):,}")
print(f"  Con V-Dem completo : {len(df) - n_sin_vdem:,} ({pct_ok:.1f}%)")
print(f"  Sin V-Dem          : {n_sin_vdem:,}")
print(f"  Columnas totales   : {df.shape[1]}")

del df_lb, df_vdem, df_vdem_raw
gc.collect()

## 6. Exclusiones iniciales

In [ ]:
# =============================================================================
# Exclusiones de variables y registros
# =============================================================================

# Paso 1: variables LB descartadas
excl = [c for c in VARS_EXCLUIR_LB if c in df.columns]
df   = df.drop(columns=excl)
print(f"Variables LB excluidas: {excl}")

# Paso 2: registros sin target válido
mask_inv = df["A_003_031"].isna() | df["A_003_031"].isin(NSNR)
n_inv    = mask_inv.sum()
df       = df[~mask_inv].copy()
print(f"Registros excluidos por target NS/NR: {n_inv:,}")

# Paso 3: Venezuela post-2017
mask_ven = (df[COL_PAIS] == "Venezuela") & (df[COL_AÑO] > AÑO_CORTE_VEN)
n_ven    = mask_ven.sum()
df       = df[~mask_ven].copy()
print(f"Registros Venezuela post-{AÑO_CORTE_VEN} excluidos: {n_ven:,}")

# Paso 4: años fuera del rango del estudio
años_validos = set()
for cfg in SUBPERIODOS.values():
    años_validos.update(cfg["train_olas"] + cfg["test_ola"])
n_antes = len(df)
df      = df[df[COL_AÑO].isin(años_validos)].copy()
print(f"Registros con año fuera del rango: {n_antes - len(df):,}")

print(f"\nDataset tras exclusiones: {df.shape[0]:,} × {df.shape[1]}")
print(f"Países : {df[COL_PAIS].nunique()}")
print(f"Años   : {sorted(df[COL_AÑO].unique())}")
gc.collect()

## 7. Recodificación de la variable dependiente

In [ ]:
# =============================================================================
# Recodificación A_003_031 → target base 0: {0, 1, 2, 3}
# =============================================================================
vals = sorted(df["A_003_031"].dropna().unique())
assert set(int(v) for v in vals) == {1, 2, 3, 4}, f"Escala inesperada: {vals}"

df[COL_TARGET] = df["A_003_031"].map({1.0:0, 2.0:1, 3.0:2, 4.0:3})
assert df[COL_TARGET].isna().sum() == 0, "NaN inesperados en target"

print("Distribución del target recodificado:")
dist = df[COL_TARGET].value_counts().sort_index()
for cls, n in dist.items():
    print(f"  Clase {int(cls)}: {n:>8,} ({n/len(df)*100:.1f}%) — {ETIQUETAS[cls]}")

# Pesos inversos de clase para manejo del desbalance sin SMOTE
PESOS_CLASE = {
    cls: len(df) / (N_CLASES * (df[COL_TARGET] == cls).sum())
    for cls in range(N_CLASES)
}
print("\nPesos inversos de clase:")
for cls, w in PESOS_CLASE.items():
    print(f"  Clase {cls} ({ETIQUETAS[cls]:<25}): {w:.4f}")

## 8. Transformaciones de escala

Se aplica la convención de **mayor valor = más favorable** de forma consistente.
El orden es crítico: NS/NR → NaN primero, luego inversiones, para evitar que
valores especiales generen datos ficticios.

In [ ]:
# =============================================================================
# Transformaciones de escala — orden obligatorio
# =============================================================================

def limpiar_nsnr(df: pd.DataFrame, cols: list, codigos: list) -> pd.DataFrame:
    """Convierte NS/NR y valores negativos a NaN. Debe ejecutarse PRIMERO."""
    df = df.copy()
    for col in cols:
        if col not in df.columns:
            continue
        mask = df[col].isin(codigos) | (pd.to_numeric(df[col], errors="coerce") < 0)
        df.loc[mask, col] = np.nan
    return df


# ── Paso 1: NS/NR → NaN en todas las columnas relevantes ─────────────────────
cols_limpiar = [c for c in df.columns
                if c not in [COL_TARGET, COL_AÑO, COL_PESO,
                              "A_003_031", COL_ISO3, COL_PAIS, "ola"]]
df = limpiar_nsnr(df, cols_limpiar, NSNR)
print("✓ NS/NR → NaN")

# ── Paso 2: Valores especiales adicionales ────────────────────────────────────
# A_007_071: código 97 = NS/NR solo en olas 2020–2024; valor 0 es válido
if "A_007_071" in df.columns:
    df.loc[df["A_007_071"] == 97, "A_007_071"] = np.nan
# C_003_003_011: código 5 = "No aplica / no trabaja"
if "C_003_003_011" in df.columns:
    df.loc[df["C_003_003_011"] == 5, "C_003_003_011"] = np.nan
# S_701: código -3 ya limpiado (no aplica por irreligiosidad — NaN)
print("✓ Valores especiales adicionales")

# ── Paso 3: Inversión Likert 4 pts — confianza institucional (5 - x) ─────────
for col in VARS_LIKERT4_INVERTIR:
    if col not in df.columns:
        continue
    mask = df[col].notna() & df[col].between(1, 4)
    df.loc[mask, col] = 5 - df.loc[mask, col]
    n_fuera = (df[col].notna() & ~df[col].between(1, 4)).sum()
    if n_fuera > 0:
        df.loc[df[col].notna() & ~df[col].between(1, 4), col] = np.nan
print(f"✓ Confianza institucional invertida ({len(VARS_LIKERT4_INVERTIR)} variables)")

# ── Paso 4: Inversión interés en política (5 - x) ────────────────────────────
for col in VARS_LIKERT4_INTERES:
    if col not in df.columns:
        continue
    mask = df[col].notna() & df[col].between(1, 4)
    df.loc[mask, col] = 5 - df.loc[mask, col]
print(f"✓ Interés en política invertido: {VARS_LIKERT4_INTERES}")

# ── Paso 5: Evaluaciones económicas comparativas ──────────────────────────────
# 1995–2000: escala 3 pts (1=Mejor → 3=Peor) → invertir (4 - x)
# 2001–2024: escala 5 pts (1=Mucho mejor → 5=Mucho peor) → invertir (6 - x)
for col in ["D_001_021", "D_001_041", "D_001_091"]:
    if col not in df.columns:
        continue
    mask3 = (df[COL_AÑO] <= 2000) & df[col].between(1, 3)
    mask5 = (df[COL_AÑO] >= 2001) & df[col].between(1, 5)
    df.loc[mask3, col] = 4 - df.loc[mask3, col]
    df.loc[mask5, col] = 6 - df.loc[mask5, col]
print("✓ Evaluaciones económicas comparativas armonizadas")

# ── Paso 6: Recodificaciones binarias ─────────────────────────────────────────
recodif = {
    "B_006_061": {1.0:1, 2.0:0},   # Aprueba=1, Desaprueba=0
    "B_001_101": {1.0:1, 2.0:0},   # País para todos=1, para poderosos=0
    "H_001_011": {1.0:1, 2.0:0},   # Confianza interpersonal: Sí=1, No=0
    "S_001"    : {1.0:0, 2.0:1},   # Sexo: Hombre=0, Mujer=1
}
for col, mapeo in recodif.items():
    if col in df.columns:
        df[col] = df[col].map(mapeo)
print(f"✓ Recodificaciones binarias: {list(recodif.keys())}")

# ── Paso 7: Victimización I_001_001 — 3 períodos de codificación ──────────────
if "I_001_001" in df.columns:
    col   = "I_001_001"
    nueva = np.full(len(df), np.nan)
    # 1995–2008: 1=Víctima, 2=No víctima
    m_bin = (df[COL_AÑO] <= 2008).values
    nueva[m_bin & (df[col].values == 1)] = 1
    nueva[m_bin & (df[col].values == 2)] = 0
    # 2009: 1=Entrevistado, 2=Familiar, 3=No
    m_09 = (df[COL_AÑO] == 2009).values
    nueva[m_09 & np.isin(df[col].values, [1, 2])] = 1
    nueva[m_09 & (df[col].values == 3)]            = 0
    # 2010–2024: 1=Entrevistado, 2=Familiar, 3=Ambos, 4=No
    m_post = (df[COL_AÑO] >= 2010).values
    nueva[m_post & np.isin(df[col].values, [1, 2, 3])] = 1
    nueva[m_post & (df[col].values == 4)]               = 0
    df[col] = nueva
print("✓ Victimización recodificada (3 períodos → binaria)")

# ── Paso 8: G_002_011 — corrupción experiencial (LAT2013 con escala anómala) ──
if "G_002_011" in df.columns:
    col = "G_002_011"
    df.loc[(df[COL_AÑO] != 2013) & (df[col] == 1), col] = 1
    df.loc[(df[COL_AÑO] != 2013) & (df[col] == 2), col] = 0
    df.loc[(df[COL_AÑO] == 2013) & (df[col] == 1), col] = 1
    df.loc[(df[COL_AÑO] == 2013) & (df[col] >  1), col] = 0
print("✓ Corrupción experiencial recodificada")

# ── Paso 9: V-Dem — índices de corrupción (1 - x) ────────────────────────────
for col in VARS_VDEM_INVERTIR:
    if col in df.columns:
        df[col] = 1.0 - df[col]
print(f"✓ Índices V-Dem invertidos: {VARS_VDEM_INVERTIR}")
print("  Semántica post-inversión: mayor valor = menor corrupción")

print()
print("=" * 55)
print(f"  Dataset final: {df.shape[0]:,} × {df.shape[1]}")
print(f"  Missingness global: {df.isnull().mean().mean()*100:.1f}%")

## EDA — Parte 1: Exploración del dataset integrado

Análisis sobre el dataset completo post-transformación y pre-split.
Objetivo: comprender la distribución del target, el patrón de missingness
y las características globales antes de particionar en subperiodos.

In [ ]:
# =============================================================================
# EDA 1.1 — Distribución del target por subperiodo histórico
# =============================================================================
SP_HIST = {
    "SP1\n1995–2007": [1995,1996,1997,1998,2000,2001,2002,2003,2004,2005,2006,2007],
    "SP2\n2008–2018": [2008,2009,2010,2011,2013,2015,2016,2017,2018],
    "SP3\n2020–2024": [2020,2023,2024],
}
colores_cls = [THEME["target"][i] for i in range(4)]

fig, axes = plt.subplots(1, len(SP_HIST) + 1, figsize=(16, 4))

# Panel global
ax = axes[0]
vals_g = df[COL_TARGET].value_counts(normalize=True).sort_index() * 100
etiq_cortas = [v.split()[0] for v in ETIQUETAS.values()]
ax.bar(etiq_cortas, vals_g.values, color=colores_cls, edgecolor="white", linewidth=0.5)
ax.set_title("Global", fontweight="bold", fontsize=11)
ax.set_ylabel("% registros")
ax.set_ylim(0, 55)
for i, v in enumerate(vals_g.values):
    ax.text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=9)

# Paneles por subperiodo
for ax, (sp_label, años) in zip(axes[1:], SP_HIST.items()):
    sub  = df[df[COL_AÑO].isin(años)]
    vals = sub[COL_TARGET].value_counts(normalize=True).sort_index() * 100
    neta = (vals.get(2, 0) + vals.get(3, 0)) - (vals.get(0, 0) + vals.get(1, 0))
    bars = [vals.get(i, 0) for i in range(4)]
    ax.bar(etiq_cortas, bars, color=colores_cls, edgecolor="white", linewidth=0.5)
    ax.set_title(f"{sp_label}(n={len(sub):,}  neta={neta:+.1f}%)",
                 fontsize=9, fontweight="bold")
    ax.set_ylim(0, 55)
    for i, v in enumerate(bars):
        ax.text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=8)

fig.suptitle("Distribución del target por subperiodo histórico", fontweight="bold")
for ax in axes:
    ax.tick_params(axis="x", labelrotation=45)
    plt.setp(ax.get_xticklabels(), ha="right")
plt.tight_layout()
plt.savefig(CARPETA_RESULTS / "eda1_target_subperiodos.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Figura guardada: eda1_target_subperiodos.png")

In [ ]:
# =============================================================================
# EDA 1.2 — Heatmap de missingness y tabla de cobertura por variable
# =============================================================================
# Solo las features del modelo (sin columnas de metadata)
FEATURES_EDA = []
for bloque, variables in BLOQUES.items():
    if "V-Dem" not in bloque:   # LB features
        FEATURES_EDA.extend([v for v in variables if v in df.columns])
FEATURES_EDA = list(dict.fromkeys(FEATURES_EDA))  # sin duplicados

fig, ax = plt.subplots(figsize=(14, 5))
msno.heatmap(df[FEATURES_EDA], ax=ax, fontsize=8, cmap="RdYlGn",
             labels=[ETIQUETAS_FEATURES.get(c, c) for c in FEATURES_EDA])
ax.set_title("Correlación de missingness entre variables de Latinobarómetro\n"
             "(verde = ausencias correlacionadas, rojo = patrones opuestos)",
             fontweight="bold")
# plt.tight_layout()
# plt.savefig(CARPETA_RESULTS / "eda1_missingness_heatmap.png", dpi=150, bbox_inches="tight")
save_figure("eda1_missingness_heatmap.png")
plt.show()

# Tabla de cobertura
print("Cobertura por variable (% de valores válidos):")
print(f"  {'Variable':<25} {'Etiqueta':<32} {'Bloque':<30} {'Cobertura'}")
print("  " + "─" * 95)
for col in FEATURES_EDA:
    cob = (1 - df[col].isnull().mean()) * 100
    et  = ETIQUETAS_FEATURES.get(col, col)
    bl  = bloque_de(col)
    print(f"  {col:<25} {et:<32} {bl:<30} {cob:>5.1f}%")

In [ ]:
# =============================================================================
# EDA 1.3 — Missingness por variable y por split (train/test de cada SP)
# =============================================================================
SP_SPLITS = {
    "SP1 train": [1995,1996,1997,1998,2000,2001,2002,2003,2004,2005],
    "SP1 val"  : [2006],
    "SP1 test" : [2007],
    "SP2 train": [1995,1996,1997,1998,2000,2001,2002,2003,2004,2005,2006,
                  2007,2008,2009,2010,2011,2013,2015,2016],
    "SP2 val"  : [2017],
    "SP2 test" : [2018],
    "SP3 train": [1995,1996,1997,1998,2000,2001,2002,2003,2004,2005,2006,
                  2007,2008,2009,2010,2011,2013,2015,2016,2017,2018,2020],
    "SP3 val"  : [2023],
    "SP3 test" : [2024],
}

miss_sp = {}
for sp_name, años in SP_SPLITS.items():
    sub = df[df[COL_AÑO].isin(años)][FEATURES_EDA]
    miss_sp[sp_name] = (sub.isnull().mean() * 100).round(1)

df_miss_sp = pd.DataFrame(miss_sp)
# Etiquetas cortas para el eje Y
df_miss_sp.index = [ETIQUETAS_FEATURES.get(c, c) for c in FEATURES_EDA]

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(df_miss_sp, annot=True, fmt=".0f", cmap="YlOrRd",
            linewidths=0.3, ax=ax, cbar_kws={"label": "% NaN"},
            annot_kws={"size": 8})
ax.set_title("Porcentaje de valores faltantes (%) por variable y conjunto\n"
             "Rojo intenso: variables que requieren imputación total en ese conjunto",
             fontweight="bold")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(CARPETA_RESULTS / "eda1_missingness_por_split.png", dpi=150, bbox_inches="tight")
plt.show()

# Alertas
print("Variables con 100% de faltantes en algún conjunto de prueba:")
df_miss_sp_orig = pd.DataFrame(miss_sp)  # con índice original para la búsqueda
for col in FEATURES_EDA:
    et = ETIQUETAS_FEATURES.get(col, col)
    tests_100 = [sp for sp in ["SP1 val","SP1 test","SP2 val","SP2 test","SP3 val","SP3 test"]
                 if df_miss_sp_orig.loc[col, sp] == 100.0]
    if tests_100:
        print(f"  {col} ({et}): 100% NaN en {tests_100}")
print("  (Estas variables se imputan completamente con la media/moda del train)")

## 9. Definición del conjunto de features por modelo

Los índices V-Dem se diferencian por tipo de modelo:
- **OLO (baseline)**: recibe los 5 índices de alto nivel, que representan
  la práctica estándar en ciencias políticas cuantitativas. Los índices
  de nivel medio presentan correlaciones de 0.70–0.97 entre sí, lo que
  generaría estimaciones inestables en la regresión logística.
- **Árboles y TabNet**: reciben los 14 índices de nivel medio más
  `v2x_polyarchy`, que aporta la dimensión democrática agregada y
  mejora la comparabilidad directa con OLO.

In [ ]:
# =============================================================================
# Conjuntos de features por tipo de modelo
# Los bloques temáticos ya están definidos en BLOQUES (sección de parámetros).
# Aquí se construyen las listas ordenadas para cada modelo.
# =============================================================================

VDEM_HIGH   = [c for c in BLOQUES["Contexto democrático · High-level"]
               if c in df.columns]

VDEM_MID    = [c for c in BLOQUES["Contexto democrático · Mid-level"]
               if c in df.columns]

LB_FEATURES = []
for bloque in ["Confianza institucional", "Evaluación económica",
               "Percepción política", "Corrupción y seguridad",
               "Características sociodemográficas"]:
    LB_FEATURES.extend([c for c in BLOQUES[bloque] if c in df.columns])

VARS_CAT    = [c for c in VARS_CATEGORICAS if c in df.columns]

FEATURES_OLO   = LB_FEATURES + VDEM_HIGH
FEATURES_TREES = LB_FEATURES + VDEM_MID   # XGBoost, CatBoost, LightGBM, TabNet

print(f"Features OLO            : {len(FEATURES_OLO)}")
print(f"Features Árboles/TabNet : {len(FEATURES_TREES)}")
print()
print("Distribución por bloque temático:")
for bloque, variables in BLOQUES.items():
    presentes = [v for v in variables if v in df.columns]
    print(f"  {bloque:<40}: {len(presentes)} variables")
print()
print(f"Variables categóricas para CatBoost: {VARS_CAT}")

## EDA — Parte 2: Validación post-transformación y selección de features

Se verifica que las transformaciones produjeron la dirección esperada y se
realiza el análisis de correlación **por subperiodo histórico**, no global.

> **Criterio de selección:** una feature se documenta como sin aporte si
> no tiene señal estadísticamente significativa en **ningún** subperiodo
> ni en ninguna subregión, Y carece de justificación teórica sustantiva.
> La correlación global puede enmascarar señal real cuando las relaciones
> cambian de dirección entre períodos históricos.

In [ ]:
# =============================================================================
# EDA 2.1 — Verificación de inversiones: correlaciones post-transformación
# =============================================================================
VARS_VERIFICAR = {
    "H_002_011" : "Confianza Congreso (invertida: mayor = más confianza)",
    "D_001_001" : "Situación económica país (mayor = mejor)",
    "B_006_061" : "Aprobación gobierno (1=aprueba, 0=desaprueba)",
    "A_007_071" : "Escala Izq-Der (0=extrema izq, 10=extrema der)",
    "v2x_corr"  : "Integridad institucional (invertida: mayor = menos corrupción)",
}

fig, axes = plt.subplots(1, len(VARS_VERIFICAR), figsize=(16, 3.5))
for ax, (col, titulo) in zip(axes, VARS_VERIFICAR.items()):
    if col not in df.columns:
        ax.set_visible(False)
        continue
    x = pd.to_numeric(df[col], errors="coerce").dropna()
    y = df.loc[x.index, COL_TARGET]
    mask = y.notna()
    r, _ = stats.spearmanr(x[mask].values, y[mask].values)
    ax.hist(x.values, bins=20, color="#2E74B5", edgecolor="white",
            linewidth=0.4, density=True)
    ax.set_title(f"{titulo}\nr Spearman = {r:+.3f}", fontsize=8.5, fontweight="bold")
    ax.set_xlabel(col, fontsize=8)

fig.suptitle("Distribuciones post-transformación — verificación de dirección",
             fontweight="bold")
plt.tight_layout()
plt.savefig(CARPETA_RESULTS / "eda2_distribuciones_post_transform.png",
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# =============================================================================
# EDA 2.2 — Correlación de Spearman por subperiodo (criterio de selección)
#
# La correlación se calcula sobre el TRAIN de cada subperiodo para evitar
# cualquier contaminación del conjunto de test.
# =============================================================================
SP_TRAIN = {
    "SP1": [1995,1996,1997,1998,2000,2001,2002,2003,2004,2005],
    "SP2": [1995,1996,1997,1998,2000,2001,2002,2003,2004,2005,2006,
            2007,2008,2009,2010,2011,2013,2015,2016],
    "SP3": [1995,1996,1997,1998,2000,2001,2002,2003,2004,2005,2006,
            2007,2008,2009,2010,2011,2013,2015,2016,2017,2018,2020],
}

def corr_sp(feat, años):
    sub  = df[df[COL_AÑO].isin(años)].copy()
    x    = pd.to_numeric(sub[feat], errors="coerce")
    y    = sub[COL_TARGET]
    mask = x.notna() & y.notna()
    if mask.sum() < 50:
        return np.nan
    r, _ = stats.spearmanr(x[mask].values, y[mask].values)
    return r

print(f"{'Feature':<22} {'Etiqueta':<30} {'Bloque':<28} {'SP1':>7} {'SP2':>7} {'SP3':>7} {'MaxSP':>7}")
print("─" * 115)

REPORTE = {}
for feat in LB_FEATURES:
    if feat not in df.columns:
        continue
    rs     = {sp: corr_sp(feat, años) for sp, años in SP_TRAIN.items()}
    valid  = [r for r in rs.values() if not np.isnan(r)]
    max_sp = max(abs(r) for r in valid) if valid else 0
    signs  = [1 if r > 0.02 else (-1 if r < -0.02 else 0) for r in valid]
    inv    = len(set(s for s in signs if s != 0)) > 1
    REPORTE[feat] = {"rs": rs, "max_sp": max_sp, "inv": inv}

    et = ETIQUETAS_FEATURES.get(feat, feat)
    bl = bloque_de(feat)[:26]
    r1 = f"{rs['SP1']:+.3f}" if not np.isnan(rs["SP1"]) else "  NaN"
    r2 = f"{rs['SP2']:+.3f}" if not np.isnan(rs["SP2"]) else "  NaN"
    r3 = f"{rs['SP3']:+.3f}" if not np.isnan(rs["SP3"]) else "  NaN"
    inv_flag = " ↕" if inv else ""
    print(f"{feat:<22} {et:<30} {bl:<28} {r1:>7} {r2:>7} {r3:>7} {max_sp:>7.3f}{inv_flag}")

print()
n_clara = sum(1 for v in REPORTE.values() if v["max_sp"] >= 0.05)
n_debil = sum(1 for v in REPORTE.values() if 0.02 <= v["max_sp"] < 0.05)
n_no    = sum(1 for v in REPORTE.values() if v["max_sp"] < 0.02)
print(f"Señal clara  (max|r|≥0.05): {n_clara} variables")
print(f"Señal débil  (max|r|≥0.02): {n_debil} variables")
print(f"Sin señal    (max|r|<0.02): {n_no} variables")
print()
print("↕ = la relación con el target cambia de signo entre subperiodos")
print("    (estas variables son especialmente relevantes para el análisis de estabilidad)")

In [ ]:
# =============================================================================
# EDA 2.3 — Heatmap de correlaciones por subperiodo, organizado por bloques
# =============================================================================
# Construir matriz de correlaciones con variables ordenadas por bloque temático
vars_ordenadas = []
bloques_labels = []
for bloque in ["Confianza institucional","Evaluación económica",
               "Percepción política","Corrupción y seguridad",
               "Características sociodemográficas"]:
    for v in BLOQUES[bloque]:
        if v in df.columns:
            vars_ordenadas.append(v)
            bloques_labels.append(bloque)

# Calcular correlaciones para cada variable en el orden definido
rs_data = {}
for feat in vars_ordenadas:
    rs_data[feat] = {
        sp: corr_sp(feat, años) for sp, años in SP_TRAIN.items()
    }

df_rs = pd.DataFrame(rs_data, index=["SP1","SP2","SP3"]).T
# Etiquetas cortas para eje Y
df_rs.index = [ETIQUETAS_FEATURES.get(c, c) for c in vars_ordenadas]

# Colores por bloque para el margen izquierdo
colores_bloques = {
    "Confianza institucional"        : "#1E3A5F",
    "Evaluación económica"           : "#0D9488",
    "Percepción política"            : "#2E74B5",
    "Corrupción y seguridad"         : "#DC2626",
    "Características sociodemográficas": "#78716C",
}

fig, (ax_bar, ax_heat) = plt.subplots(
    1, 2, figsize=(13, 10),
    gridspec_kw={"width_ratios": [0.04, 0.96]},
)

# Barra lateral de bloques
for i, (feat, bloque) in enumerate(zip(vars_ordenadas, bloques_labels)):
    ax_bar.barh(i, 1, color=colores_bloques[bloque], edgecolor="none")
ax_bar.set_xlim(0, 1)
ax_bar.set_ylim(-0.5, len(vars_ordenadas) - 0.5)
ax_bar.axis("off")

# Leyenda de bloques
patches = [mpatches.Patch(color=c, label=b)
           for b, c in colores_bloques.items()]
ax_bar.legend(handles=patches, loc="upper left",
              bbox_to_anchor=(0, -0.02), fontsize=8, frameon=True)

# Heatmap principal
sns.heatmap(
    df_rs, annot=True, fmt=".2f", cmap="RdYlGn",
    center=0, vmin=-0.45, vmax=0.45,
    linewidths=0.3, ax=ax_heat,
    annot_kws={"size": 8},
    cbar_kws={"label": "r Spearman", "shrink": 0.6},
)
ax_heat.set_title(
    "Correlación Spearman con satisfacción democrática por subperiodo\n"
    "(calculada sobre el train de cada subperiodo — sin contaminación del test)",
    fontweight="bold", pad=12,
)
ax_heat.set_xlabel("Subperiodo")
ax_heat.set_ylabel("")
ax_heat.tick_params(axis="y", labelsize=9)

# Líneas separadoras entre bloques
n_acum = 0
for bloque in ["Confianza institucional","Evaluación económica",
               "Percepción política","Corrupción y seguridad"]:
    n_acum += len([v for v in BLOQUES[bloque] if v in df.columns])
    ax_heat.axhline(n_acum, color="white", linewidth=2)

plt.tight_layout()
plt.savefig(CARPETA_RESULTS / "eda2_correlaciones_subperiodo.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("✓ Heatmap de correlaciones por subperiodo guardado")

## 10. Construcción de los splits temporales (Expanding Window)

In [ ]:
# =============================================================================
# Función de construcción de splits — Expanding Window Walk-Forward
# =============================================================================

def construir_split(
    df: pd.DataFrame,
    subperiodo: str,
    features: list,
) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series,
           pd.DataFrame, pd.Series, np.ndarray, np.ndarray, np.ndarray]:
    """
    Construye train / val / test para un subperiodo dado.
    El sample_weight combina el factor de expansión muestral (X_020)
    con los pesos inversos de clase para el manejo del desbalance.
    """
    cfg   = SUBPERIODOS[subperiodo]
    feats = [f for f in features if f in df.columns and f != COL_PESO]

    df_tr  = df[df[COL_AÑO].isin(cfg["train_olas"])].copy()
    df_val = df[df[COL_AÑO].isin(cfg["validate_ola"])].copy()
    df_te  = df[df[COL_AÑO].isin(cfg["test_ola"])].copy()

    X_tr,  y_tr  = df_tr[feats],  df_tr[COL_TARGET].astype(int)
    X_val, y_val = df_val[feats], df_val[COL_TARGET].astype(int)
    X_te,  y_te  = df_te[feats],  df_te[COL_TARGET].astype(int)

    def _pesos(df_sub, y):
        w_m = (df_sub[COL_PESO].fillna(1.0)
               if COL_PESO in df_sub.columns
               else pd.Series(np.ones(len(df_sub)), index=df_sub.index))
        w_m = w_m / w_m.mean()
        w_c = y.map(PESOS_CLASE)
        return (w_m.values * w_c.values).astype(float)

    w_tr  = _pesos(df_tr, y_tr)
    w_val = _pesos(df_val, y_val)
    w_te  = _pesos(df_te, y_te)
    return X_tr, y_tr, X_val, y_val, X_te, y_te, w_tr, w_val, w_te


def resumen_split(nombre, X_tr, y_tr, X_val, y_val, X_te, y_te):
    print(f"{'─'*52}")
    print(f"  {nombre}")
    print(f"{'─'*52}")
    print(f"  Train : {len(X_tr):>8,} registros | {X_tr.shape[1]} features")
    print(f"  Val   : {len(X_val):>8,} registros")
    print(f"  Test  : {len(X_te):>8,} registros")
    print(f"  Ratio train/test: {len(X_tr)/len(X_te):.1f}x")
    print(f"  Clases train : {dict(y_tr.value_counts().sort_index())}")
    print(f"  Clases val   : {dict(y_val.value_counts().sort_index())}")
    print(f"  Clases test  : {dict(y_te.value_counts().sort_index())}")
    miss_tr  = X_tr.isnull().mean().mean() * 100
    miss_val = X_val.isnull().mean().mean() * 100
    miss_te  = X_te.isnull().mean().mean() * 100
    print(f"  NaN train: {miss_tr:.1f}%  |  NaN val: {miss_val:.1f}%  |  NaN test: {miss_te:.1f}%")


print("✓ Funciones de split definidas.")

## 11. Imputación diferenciada y normalización

Estrategia por tipo de modelo:
- **OLO y TabNet**: MICE (`IterativeImputer` + `BayesianRidge`) para numéricas,
  moda para categóricas. Requieren matrices completas.
- **XGBoost, LightGBM, CatBoost**: reciben datos con NaN nativos; cada
  algoritmo tiene su propio mecanismo interno de manejo de valores faltantes.

> **Anti-data leakage:** el imputador se ajusta exclusivamente sobre el
> conjunto de entrenamiento de cada split.

In [ ]:
# =============================================================================
# Funciones de imputación y normalización
# =============================================================================

def imputar(
    X_tr: pd.DataFrame,
    X_val: pd.DataFrame,
    X_te: pd.DataFrame,
    semilla: int = SEED,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, IterativeImputer, SimpleImputer]:
    """
    MICE para columnas numéricas, moda para categóricas.
    Ajuste exclusivo sobre X_tr; solo transform sobre X_val y X_te.
    Retorna (X_tr_imp, X_val_imp, X_te_imp, imputer_num, imputer_cat).
    """
    cols_cat = [c for c in VARS_CAT if c in X_tr.columns]
    cols_num = [c for c in X_tr.columns if c not in cols_cat]

    imp_num = IterativeImputer(
        estimator=BayesianRidge(), max_iter=10,
        random_state=semilla, verbose=0,
    )
    X_tr_num  = pd.DataFrame(imp_num.fit_transform(X_tr[cols_num]),
                              columns=cols_num, index=X_tr.index)
    X_val_num = pd.DataFrame(imp_num.transform(X_val[cols_num]),
                              columns=cols_num, index=X_val.index)
    X_te_num  = pd.DataFrame(imp_num.transform(X_te[cols_num]),
                              columns=cols_num, index=X_te.index)

    imp_cat = SimpleImputer(strategy="most_frequent")
    if cols_cat:
        X_tr_cat  = pd.DataFrame(imp_cat.fit_transform(X_tr[cols_cat]),
                                  columns=cols_cat, index=X_tr.index)
        X_val_cat = pd.DataFrame(imp_cat.transform(X_val[cols_cat]),
                                  columns=cols_cat, index=X_val.index)
        X_te_cat  = pd.DataFrame(imp_cat.transform(X_te[cols_cat]),
                                  columns=cols_cat, index=X_te.index)
        X_tr_imp  = pd.concat([X_tr_num,  X_tr_cat],  axis=1)[X_tr.columns]
        X_val_imp = pd.concat([X_val_num, X_val_cat], axis=1)[X_val.columns]
        X_te_imp  = pd.concat([X_te_num,  X_te_cat],  axis=1)[X_te.columns]
    else:
        X_tr_imp, X_val_imp, X_te_imp = X_tr_num, X_val_num, X_te_num

    assert X_tr_imp.isnull().sum().sum()  == 0, "NaN residuales tras imputación (train)"
    assert X_val_imp.isnull().sum().sum() == 0, "NaN residuales tras imputación (val)"
    assert X_te_imp.isnull().sum().sum()  == 0, "NaN residuales tras imputación (test)"

    return X_tr_imp, X_val_imp, X_te_imp, imp_num, imp_cat


def normalizar(
    X_tr: pd.DataFrame,
    X_val: pd.DataFrame,
    X_te: pd.DataFrame,
    metodo: str = "minmax",
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, object]:
    """'minmax' para TabNet | 'standard' para OLO"""
    cols_num = [c for c in X_tr.columns if c not in VARS_CAT]
    scaler   = MinMaxScaler() if metodo == "minmax" else StandardScaler()
    X_tr_sc  = X_tr.copy()
    X_val_sc = X_val.copy()
    X_te_sc  = X_te.copy()
    X_tr_sc[cols_num]  = scaler.fit_transform(X_tr[cols_num])
    X_val_sc[cols_num] = scaler.transform(X_val[cols_num])
    X_te_sc[cols_num]  = scaler.transform(X_te[cols_num])
    return X_tr_sc, X_val_sc, X_te_sc, scaler


print("✓ Funciones de imputación y normalización definidas.")

## 12. Función de evaluación unificada

In [ ]:
# =============================================================================
# Función de evaluación — métricas para clasificación ordinal multiclase
# =============================================================================

def evaluar(
    y_true, y_pred, y_prob=None, nombre="", subperiodo="", split="test",
) -> Dict:
    metricas = {
        "modelo"           : nombre,
        "subperiodo"       : subperiodo,
        "split"            : split,
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1_macro"         : f1_score(y_true, y_pred, average="macro",    zero_division=0),
        "f1_weighted"      : f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "kappa_lineal"     : cohen_kappa_score(y_true, y_pred, weights="linear"),
        "kappa_cuadratico" : cohen_kappa_score(y_true, y_pred, weights="quadratic"),
        "mae_ordinal"      : mean_absolute_error(y_true, y_pred),
        "auroc_macro"      : np.nan,
    }
    if y_prob is not None:
        try:
            metricas["auroc_macro"] = roc_auc_score(
                y_true, y_prob, multi_class="ovr", average="macro"
            )
        except Exception:
            pass
    if nombre:
        print(f"  {'─'*48}")
        print(f"  {nombre} — {subperiodo} [{split}]")
        print(f"  {'─'*48}")
        for k, v in metricas.items():
            if k in ("modelo", "subperiodo"):
                continue
            print(f"    {k:<22}: {v:.4f}" if isinstance(v, float) else f"    {k}: {v}")
    return metricas

RESULTADOS: List[Dict] = []
print("✓ Función de evaluación definida.")

## 13. Modelo 1 — Regresión Logística Ordinal (Baseline)

In [ ]:
# =============================================================================
# Regresión Logística Ordinal — mord.LogisticIT
# Fallback: LogisticRegression con solver='lbfgs' (multinomial por defecto)
# =============================================================================
try:
    import mord
    MORD_OK = True
    print("✓ mord disponible")
except ImportError:
    MORD_OK = False
    print("⚠ mord no instalado — usando LogisticRegression(solver='lbfgs')")


def entrenar_olo(X_tr, y_tr, X_val, y_val, X_te, y_te, w_tr, w_val, sp):
    nombre = "OLO"
    print(f"{'='*52}  Entrenando {nombre} — {sp}{'='*52}")
    ruta_hp = CARPETA_MODELS / f"hp_{nombre}_{sp}.json"

    if not EJECUTAR_BUSQUEDA_HP and ruta_hp.exists():
        best_hp = json.loads(ruta_hp.read_text())
        print(f"  HPs cargados: {best_hp}")
    else:
        X_tr_np,  y_tr_np  = np.array(X_tr),  np.array(y_tr)
        X_val_np, y_val_np = np.array(X_val), np.array(y_val)
        def obj(trial):
            alpha = trial.suggest_float("alpha", 1e-4, 10.0, log=True)
            if MORD_OK:
                m = mord.LogisticIT(alpha=alpha, max_iter=500)
                m.fit(X_tr_np, y_tr_np, sample_weight=w_tr)
            else:
                m = LogisticRegression(C=1/alpha, solver="lbfgs",
                                       max_iter=500, random_state=SEED)
                m.fit(X_tr_np, y_tr_np, sample_weight=w_tr)
            return cohen_kappa_score(y_val_np, m.predict(X_val_np),
                                     weights="quadratic")
        study = optuna.create_study(direction="maximize",
                                    sampler=TPESampler(seed=SEED))
        study.optimize(obj, n_trials=N_TRIALS_OPTUNA, show_progress_bar=False)
        best_hp = study.best_params
        print(f"  Mejor Kappa Val: {study.best_value:.4f} | {best_hp}")
        ruta_hp.write_text(json.dumps(best_hp))

    alpha = best_hp.get("alpha", 1.0)
    if MORD_OK:
        clf = mord.LogisticIT(alpha=alpha, max_iter=500)
        clf.fit(np.array(X_tr), np.array(y_tr), sample_weight=w_tr)
    else:
        clf = LogisticRegression(C=1/alpha, solver="lbfgs",
                                 max_iter=500, random_state=SEED)
        clf.fit(np.array(X_tr), np.array(y_tr), sample_weight=w_tr)

    y_pred_val = clf.predict(np.array(X_val))
    y_prob_val = clf.predict_proba(np.array(X_val))
    y_pred_te  = clf.predict(np.array(X_te))
    y_prob_te  = clf.predict_proba(np.array(X_te))

    joblib.dump(clf, CARPETA_MODELS / f"{nombre}_{sp}.pkl")
    print(f"  ✓ Guardado: {nombre}_{sp}.pkl")
    m_val = evaluar(y_val, y_pred_val, y_prob_val, nombre, sp, split="val")
    m_te  = evaluar(y_te,  y_pred_te,  y_prob_te,  nombre, sp, split="test")
    RESULTADOS.extend([m_val, m_te])
    return clf, m_te

print("✓ Función OLO definida.")

## 14. Modelo 2 — XGBoost

In [ ]:
# =============================================================================
# XGBoost — manejo nativo de NaN; sample_weight para desbalance
# =============================================================================

def entrenar_xgboost(X_tr, y_tr, X_val, y_val, X_te, y_te, w_tr, w_val, sp):
    nombre = "XGBoost"
    print(f"{'='*52}  Entrenando {nombre} — {sp}{'='*52}")
    ruta_hp = CARPETA_MODELS / f"hp_{nombre}_{sp}.json"

    if not EJECUTAR_BUSQUEDA_HP and ruta_hp.exists():
        best_hp = json.loads(ruta_hp.read_text())
        print(f"  HPs cargados: {best_hp}")
    else:
        def obj(trial):
            p = {
                "n_estimators"    : trial.suggest_int("n_estimators", 200, 1000, step=100),
                "max_depth"       : trial.suggest_int("max_depth", 3, 8),
                "learning_rate"   : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "subsample"       : trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
                "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
                "reg_alpha"       : trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
                "reg_lambda"      : trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
                "objective": "multi:softprob", "num_class": N_CLASES,
                "tree_method": "hist",
                "device": "cuda" if USAR_GPU else "cpu",
                "random_state": SEED, "n_jobs": N_JOBS, "verbosity": 0,
            }
            clf = xgb.XGBClassifier(**p)
            clf.fit(X_tr, y_tr, sample_weight=w_tr,
                    eval_set=[(X_val, y_val)], verbose=False)
            return cohen_kappa_score(y_val, clf.predict(X_val),
                                     weights="quadratic")
        study = optuna.create_study(direction="maximize",
                                    sampler=TPESampler(seed=SEED))
        study.optimize(obj, n_trials=N_TRIALS_OPTUNA, show_progress_bar=False)
        best_hp = study.best_params
        print(f"  Mejor Kappa Val: {study.best_value:.4f} | {best_hp}")
        ruta_hp.write_text(json.dumps(best_hp))

    clf = xgb.XGBClassifier(
        **best_hp,
        objective="multi:softprob", num_class=N_CLASES,
        tree_method="hist",
        device="cuda" if USAR_GPU else "cpu",
        random_state=SEED, n_jobs=N_JOBS, verbosity=0,
    )
    clf.fit(X_tr, y_tr, sample_weight=w_tr,
            eval_set=[(X_val, y_val)], verbose=False)

    y_pred_val = clf.predict(X_val)
    y_prob_val = clf.predict_proba(X_val)
    y_pred_te  = clf.predict(X_te)
    y_prob_te  = clf.predict_proba(X_te)

    joblib.dump(clf, CARPETA_MODELS / f"{nombre}_{sp}.pkl")
    print(f"  ✓ Guardado: {nombre}_{sp}.pkl")
    m_val = evaluar(y_val, y_pred_val, y_prob_val, nombre, sp, split="val")
    m_te  = evaluar(y_te,  y_pred_te,  y_prob_te,  nombre, sp, split="test")
    RESULTADOS.extend([m_val, m_te])
    return clf, m_te

print("✓ Función XGBoost definida.")

## 15. Modelo 3 — CatBoost

In [ ]:
# =============================================================================
# CatBoost — categóricas nativas; manejo interno de NaN
# =============================================================================

def entrenar_catboost(X_tr, y_tr, X_val, y_val, X_te, y_te, w_tr, w_val, sp):
    nombre = "CatBoost"
    print(f"\n{'='*52}\n  Entrenando {nombre} — {sp}\n{'='*52}")

    def prep_cat(X):
        X = X.copy()
        for col in VARS_CAT:
            if col in X.columns:
                X[col] = X[col].fillna(-999).astype(int).astype(str)
        return X

    X_tr_c, X_val_c, X_te_c = prep_cat(X_tr), prep_cat(X_val), prep_cat(X_te)
    cat_idx = [i for i, c in enumerate(X_tr.columns) if c in VARS_CAT]

    ruta_hp = CARPETA_MODELS / f"hp_{nombre}_{sp}.json"
    if not EJECUTAR_BUSQUEDA_HP and ruta_hp.exists():
        best_hp = json.loads(ruta_hp.read_text())
        print(f"  HPs cargados: {best_hp}")
    else:
        def obj(trial):
            p = {
                "iterations"         : trial.suggest_int("iterations", 300, 1000, step=100),
                "depth"              : trial.suggest_int("depth", 4, 8),
                "learning_rate"      : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "l2_leaf_reg"        : trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
                "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
                "border_count"       : trial.suggest_int("border_count", 32, 128),
                "random_strength"    : trial.suggest_float("random_strength", 0.0, 10.0),
            }
            pool_tr  = Pool(X_tr_c, label=y_tr.values, weight=w_tr, cat_features=cat_idx)
            pool_val = Pool(X_val_c, label=y_val.values, cat_features=cat_idx)
            clf = CatBoostClassifier(**p, loss_function="MultiClass",
                                     random_seed=SEED, verbose=False,
                                     task_type="GPU" if USAR_GPU else "CPU")
            clf.fit(pool_tr, eval_set=pool_val)
            return cohen_kappa_score(y_val, clf.predict(X_val_c).flatten(),
                                     weights="quadratic")
        study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=SEED))
        study.optimize(obj, n_trials=N_TRIALS_OPTUNA, show_progress_bar=False)
        best_hp = study.best_params
        print(f"  Mejor Kappa Val: {study.best_value:.4f} | {best_hp}")
        ruta_hp.write_text(json.dumps(best_hp))

    pool_final = Pool(X_tr_c, label=y_tr.values, weight=w_tr, cat_features=cat_idx)
    pool_val   = Pool(X_val_c, label=y_val.values, cat_features=cat_idx)
    clf = CatBoostClassifier(**best_hp, loss_function="MultiClass",
                              random_seed=SEED, verbose=False,
                              task_type="GPU" if USAR_GPU else "CPU")
    clf.fit(pool_final, eval_set=pool_val)

    y_pred_val = clf.predict(X_val_c).flatten()
    y_prob_val = clf.predict_proba(X_val_c)
    y_pred_te  = clf.predict(X_te_c).flatten()
    y_prob_te  = clf.predict_proba(X_te_c)

    clf.save_model(str(CARPETA_MODELS / f"{nombre}_{sp}.cbm"))
    print(f"  ✓ Guardado: {nombre}_{sp}.cbm")
    m_val = evaluar(y_val, y_pred_val, y_prob_val, nombre, sp, split="val")
    m_te  = evaluar(y_te,  y_pred_te,  y_prob_te,  nombre, sp, split="test")
    RESULTADOS.extend([m_val, m_te])
    return clf, m_te

print("✓ Función CatBoost definida.")

## 16. Modelo 4 — LightGBM

In [ ]:
# =============================================================================
# LightGBM — manejo nativo de NaN; categóricas como dtype 'category'
# =============================================================================

def entrenar_lightgbm(X_tr, y_tr, X_val, y_val, X_te, y_te, w_tr, w_val, sp):
    nombre = "LightGBM"
    print(f"\n{'='*52}\n  Entrenando {nombre} — {sp}\n{'='*52}")

    def prep_lgb(Xa, Xb, Xc):
        Xa, Xb, Xc = Xa.copy(), Xb.copy(), Xc.copy()
        for col in VARS_CAT:
            if col not in Xa.columns:
                continue
            cats = sorted(set(Xa[col].dropna().tolist() +
                              Xb[col].dropna().tolist() +
                              Xc[col].dropna().tolist()))
            ct = pd.CategoricalDtype(categories=cats, ordered=False)
            Xa[col] = Xa[col].astype(ct)
            Xb[col] = Xb[col].astype(ct)
            Xc[col] = Xc[col].astype(ct)
        return Xa, Xb, Xc

    X_tr_l, X_val_l, X_te_l = prep_lgb(X_tr, X_val, X_te)

    ruta_hp = CARPETA_MODELS / f"hp_{nombre}_{sp}.json"
    if not EJECUTAR_BUSQUEDA_HP and ruta_hp.exists():
        best_hp = json.loads(ruta_hp.read_text())
        print(f"  HPs cargados: {best_hp}")
    else:
        def obj(trial):
            p = {
                "n_estimators"     : trial.suggest_int("n_estimators", 200, 1000, step=100),
                "num_leaves"       : trial.suggest_int("num_leaves", 20, 150),
                "max_depth"        : trial.suggest_int("max_depth", 3, 8),
                "learning_rate"    : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
                "subsample"        : trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.6, 1.0),
                "reg_alpha"        : trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
                "reg_lambda"       : trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
                "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
            }
            clf = lgb.LGBMClassifier(
                **p, objective="multiclass", num_class=N_CLASES,
                class_weight=PESOS_CLASE, random_state=SEED,
                n_jobs=N_JOBS, verbose=-1,
                device="cuda" if USAR_GPU else "cpu",
            )
            clf.fit(X_tr_l, y_tr, sample_weight=w_tr,
                    eval_set=[(X_val_l, y_val)],
                    callbacks=[lgb.early_stopping(50, verbose=False),
                               lgb.log_evaluation(-1)])
            return cohen_kappa_score(y_val, clf.predict(X_val_l), weights="quadratic")
        study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=SEED))
        study.optimize(obj, n_trials=N_TRIALS_OPTUNA, show_progress_bar=False)
        best_hp = study.best_params
        print(f"  Mejor Kappa Val: {study.best_value:.4f} | {best_hp}")
        ruta_hp.write_text(json.dumps(best_hp))

    clf = lgb.LGBMClassifier(
        **best_hp, objective="multiclass", num_class=N_CLASES,
        class_weight=PESOS_CLASE, random_state=SEED,
        n_jobs=N_JOBS, verbose=-1,
        device="cuda" if USAR_GPU else "cpu",
    )
    clf.fit(X_tr_l, y_tr, sample_weight=w_tr,
            eval_set=[(X_val_l, y_val)],
            callbacks=[lgb.early_stopping(50, verbose=False),
                       lgb.log_evaluation(-1)])

    y_pred_val = clf.predict(X_val_l)
    y_prob_val = clf.predict_proba(X_val_l)
    y_pred_te  = clf.predict(X_te_l)
    y_prob_te  = clf.predict_proba(X_te_l)

    joblib.dump(clf, CARPETA_MODELS / f"{nombre}_{sp}.pkl")
    print(f"  ✓ Guardado: {nombre}_{sp}.pkl")
    m_val = evaluar(y_val, y_pred_val, y_prob_val, nombre, sp, split="val")
    m_te  = evaluar(y_te,  y_pred_te,  y_prob_te,  nombre, sp, split="test")
    RESULTADOS.extend([m_val, m_te])
    return clf, m_te

print("✓ Función LightGBM definida.")

## 17. Modelo 5 — TabNet

In [ ]:
# =============================================================================
# TabNet — requiere matrices completas (imputadas) y escala [0,1]
# Limitación documentada: no soporta sample_weight por observación individual;
# se usan pesos por clase (weights=1 activa inverse-frequency automático).
# =============================================================================

def entrenar_tabnet(X_tr_sc, y_tr, X_val_sc, y_val, X_te_sc, y_te, sp, cat_idxs, cat_dims):
    nombre = "TabNet"
    print(f"\n{'='*52}\n  Entrenando {nombre} — {sp}\n{'='*52}")
    print(f"  Dispositivo: {DISPOSITIVO_TN}")

    ruta_hp = CARPETA_MODELS / f"hp_{nombre}_{sp}.json"
    if not EJECUTAR_BUSQUEDA_HP and ruta_hp.exists():
        best_hp = json.loads(ruta_hp.read_text())
        print(f"  HPs cargados: {best_hp}")
    else:
        def obj(trial):
            p = {
                "n_d"          : trial.suggest_int("n_d", 8, 64, step=8),
                "n_a"          : trial.suggest_int("n_a", 8, 64, step=8),
                "n_steps"      : trial.suggest_int("n_steps", 3, 7),
                "gamma"        : trial.suggest_float("gamma", 1.0, 2.0),
                "lambda_sparse": trial.suggest_float("lambda_sparse", 1e-6, 1e-3, log=True),
                "momentum"     : trial.suggest_float("momentum", 0.01, 0.4),
                "mask_type"    : trial.suggest_categorical("mask_type",
                                     ["sparsemax", "entmax"]),
            }
            lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
            clf = TabNetClassifier(
                **p,
                optimizer_fn=torch.optim.Adam,
                optimizer_params={"lr": lr},
                scheduler_fn=torch.optim.lr_scheduler.StepLR,
                scheduler_params={"step_size": 10, "gamma": 0.9},
                cat_idxs=cat_idxs, cat_dims=cat_dims, cat_emb_dim=3,
                verbose=0, device_name=DISPOSITIVO_TN, seed=SEED,
            )
            clf.fit(
                X_tr_sc.astype(np.float32), y_tr,
                eval_set=[(X_val_sc.astype(np.float32), y_val)],
                eval_metric=["balanced_accuracy"],
                max_epochs=100, patience=15,
                batch_size=1024, virtual_batch_size=128, weights=1,
            )
            return cohen_kappa_score(y_val,
                                     clf.predict(X_val_sc.astype(np.float32)),
                                     weights="quadratic")
        study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=SEED))
        study.optimize(obj, n_trials=min(N_TRIALS_OPTUNA, 20), show_progress_bar=False)
        best_hp = study.best_params
        print(f"  Mejor Kappa Val: {study.best_value:.4f} | {best_hp}")
        ruta_hp.write_text(json.dumps(best_hp))

    lr_opt = best_hp.pop("lr", 1e-3)
    clf = TabNetClassifier(
        **best_hp,
        optimizer_fn=torch.optim.Adam,
        optimizer_params={"lr": lr_opt},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        scheduler_params={"step_size": 10, "gamma": 0.9},
        cat_idxs=cat_idxs, cat_dims=cat_dims, cat_emb_dim=3,
        verbose=0, device_name=DISPOSITIVO_TN, seed=SEED,
    )
    clf.fit(
        X_tr_sc.astype(np.float32), y_tr,
        eval_set=[(X_val_sc.astype(np.float32), y_val)],
        eval_metric=["balanced_accuracy"],
        max_epochs=200, patience=20,
        batch_size=1024, virtual_batch_size=128, weights=1,
    )
    best_hp["lr"] = lr_opt

    y_pred_val = clf.predict(X_val_sc.astype(np.float32))
    y_prob_val = clf.predict_proba(X_val_sc.astype(np.float32))
    y_pred_te  = clf.predict(X_te_sc.astype(np.float32))
    y_prob_te  = clf.predict_proba(X_te_sc.astype(np.float32))

    clf.save_model(str(CARPETA_MODELS / f"{nombre}_{sp}"))
    print(f"  ✓ Guardado: {nombre}_{sp}.zip")
    print("  ⚠ Limitación: TabNet usa pesos por clase, no sample_weight individual.")
    m_val = evaluar(y_val, y_pred_val, y_prob_val, nombre, sp, split="val")
    m_te  = evaluar(y_te,  y_pred_te,  y_prob_te,  nombre, sp, split="test")
    RESULTADOS.extend([m_val, m_te])
    return clf, m_te

print("✓ Función TabNet definida.")

## 18. Ciclo principal de entrenamiento y guardado del pipeline

Por cada subperiodo se entrenan los 5 modelos y se guarda el **pipeline completo**
como artefacto autónomo que incluye el modelo, los preprocesadores ajustados y
la metadata necesaria para reproducir predicciones en producción.

In [ ]:
# =============================================================================
# Ciclo principal — SP1 → SP2 → SP3
# El pipeline completo se guarda dentro del loop para que cada subperiodo
# quede encapsulado con sus propios preprocesadores ajustados.
# =============================================================================
PIPELINES: Dict[str, Dict] = {}

for sp, cfg in SUBPERIODOS.items():
    print()
    print("#" * 60)
    print(f"# SUBPERIODO: {sp} — {cfg['descripcion']}")
    print("#" * 60)

    # ── Splits ────────────────────────────────────────────────────────────────
    X_tr_t, y_tr, X_val_t, y_val, X_te_t, y_te, w_tr, w_val, w_te = construir_split(df, sp, FEATURES_TREES)
    X_tr_o, y_tr_o, X_val_o, y_val_o, X_te_o, y_te_o, w_tr_o, w_val_o, _ = construir_split(df, sp, FEATURES_OLO)
    resumen_split(sp, X_tr_t, y_tr, X_val_t, y_val, X_te_t, y_te)

    # ── Imputación para OLO y TabNet ──────────────────────────────────────────
    print("\n  [Imputación MICE...]")
    X_tr_oi, X_val_oi, X_te_oi, imp_num_olo, imp_cat_olo = imputar(X_tr_o, X_val_o, X_te_o)
    X_tr_ti, X_val_ti, X_te_ti, imp_num_tab, imp_cat_tab = imputar(X_tr_t, X_val_t, X_te_t)

    # ── Normalización ─────────────────────────────────────────────────────────
    print("  [Normalización...]")
    X_tr_os, X_val_os, X_te_os, sc_olo = normalizar(X_tr_oi, X_val_oi, X_te_oi, "standard")
    X_tr_ts, X_val_ts, X_te_ts, sc_tab = normalizar(X_tr_ti, X_val_ti, X_te_ti, "minmax")

    # ── Índices de categóricas para TabNet ────────────────────────────────────
    cols_tn  = list(X_tr_ts.columns)
    cat_idxs = [i for i, c in enumerate(cols_tn) if c in VARS_CAT]
    cat_dims = [int(X_tr_ts[c].nunique()) + 1 for c in VARS_CAT if c in cols_tn]

    # ── Guardar datasets procesados en Parquet ────────────────────────────────
    print("\n  [Guardando datasets en Parquet...]")
    X_tr_t.assign(target=y_tr.values).to_parquet(
        CARPETA_PROC / f"{sp}_train.parquet", index=False)
    X_val_t.assign(target=y_val.values).to_parquet(
        CARPETA_PROC / f"{sp}_val.parquet", index=False)
    X_te_t.assign(target=y_te.values).to_parquet(
        CARPETA_PROC / f"{sp}_test.parquet", index=False)
    pd.DataFrame({"sample_weight": w_tr}).to_parquet(
        CARPETA_PROC / f"{sp}_train_weights.parquet", index=False)
    pd.DataFrame({"sample_weight": w_val}).to_parquet(
        CARPETA_PROC / f"{sp}_val_weights.parquet", index=False)

    # ── Entrenamiento ─────────────────────────────────────────────────────────
    clf_olo, _ = entrenar_olo(
        X_tr_os.values, y_tr_o.values,
        X_val_os.values, y_val_o.values,
        X_te_os.values, y_te_o.values, w_tr_o, w_val_o, sp)
    clf_xgb, _ = entrenar_xgboost(X_tr_t, y_tr, X_val_t, y_val, X_te_t, y_te, w_tr, w_val, sp)
    clf_cb,  _ = entrenar_catboost(X_tr_t, y_tr, X_val_t, y_val, X_te_t, y_te, w_tr, w_val, sp)
    clf_lgb, _ = entrenar_lightgbm(X_tr_t, y_tr, X_val_t, y_val, X_te_t, y_te, w_tr, w_val, sp)
    clf_tn,  _ = entrenar_tabnet(
        X_tr_ts.values, y_tr.values,
        X_val_ts.values, y_val.values,
        X_te_ts.values, y_te.values, sp, cat_idxs, cat_dims)

    # ── Pipeline completo por modelo y subperiodo ─────────────────────────────
    for nombre_m, clf_m, tipo in [
        ("OLO",     clf_olo, "olo"),
        ("XGBoost", clf_xgb, "trees"),
        ("CatBoost",clf_cb,  "trees"),
        ("LightGBM",clf_lgb, "trees"),
        ("TabNet",  clf_tn,  "tabnet"),
    ]:
        artefacto = {
            "modelo"            : clf_m,
            "tipo_modelo"       : tipo,
            "nombre_modelo"     : nombre_m,
            "subperiodo"        : sp,
            # Preprocesadores ajustados sobre train
            "imp_num"           : imp_num_olo if tipo == "olo" else (
                                  imp_num_tab if tipo == "tabnet" else None),
            "imp_cat"           : imp_cat_olo if tipo == "olo" else (
                                  imp_cat_tab if tipo == "tabnet" else None),
            "scaler"            : sc_olo      if tipo == "olo" else (
                                  sc_tab      if tipo == "tabnet" else None),
            # Features y categóricas
            "features"          : FEATURES_OLO if tipo == "olo" else FEATURES_TREES,
            "vars_categoricas"  : VARS_CAT,
            "cat_idxs_tabnet"   : cat_idxs if tipo == "tabnet" else None,
            "cat_dims_tabnet"   : cat_dims  if tipo == "tabnet" else None,
            # Etiquetas y bloques para visualizaciones externas (XAI)
            "etiquetas_features": ETIQUETAS_FEATURES,
            "bloques"           : BLOQUES,
            # Transformaciones deterministas para producción
            "transformaciones"  : {
                "nsnr"            : NSNR,
                "likert4"         : VARS_LIKERT4_INVERTIR,
                "likert4_interes" : VARS_LIKERT4_INTERES,
                "binarias"        : {
                    "B_006_061": {1:1, 2:0},
                    "B_001_101": {1:1, 2:0},
                    "H_001_011": {1:1, 2:0},
                    "S_001"    : {1:0, 2:1},
                },
                "vdem_invertir"   : VARS_VDEM_INVERTIR,
                "a007071_nsnr_97" : True,
                "c003003_nan5"    : True,
            },
            # Metadatos
            "version_pipeline"  : "2.0.0",
            "fecha_entrenamiento": datetime.now().isoformat(),
            "etiquetas_target"  : ETIQUETAS,
            "pesos_clase"       : PESOS_CLASE,
        }
        ruta_art = CARPETA_MODELS / f"pipeline_{nombre_m}_{sp}.pkl"
        joblib.dump(artefacto, ruta_art)
        PIPELINES[f"{nombre_m}_{sp}"] = artefacto
        print(f"  ✓ Pipeline guardado: pipeline_{nombre_m}_{sp}.pkl")

    gc.collect()
    print(f"\n✓ {sp} completado.")

print()
print("=" * 60)
print("✅ CICLO COMPLETO FINALIZADO")
print(f"   Pipelines guardados: {len(PIPELINES)}")
print("=" * 60)

## 19. Función de predicción en producción

La función `predecir()` encapsula el pipeline completo para cualquier modelo.
Recibe los valores **originales del cuestionario** y aplica internamente
todas las transformaciones necesarias en el orden correcto.

In [ ]:
# =============================================================================
# Transformaciones deterministas — reutilizadas en entrenamiento y producción
# =============================================================================

def aplicar_transformaciones_deterministas(
    df_in: pd.DataFrame,
    transformaciones: dict,
    año_encuesta: int,
) -> pd.DataFrame:
    """
    Aplica todas las transformaciones de escala en el orden obligatorio.
    Esta función es idéntica a la utilizada en la sección 8 del pipeline
    de entrenamiento, garantizando consistencia entre entrenamiento y producción.

    Parámetros
    ----------
    df_in         : DataFrame con los valores originales del cuestionario.
    transformaciones: dict extraído del artefacto del pipeline.
    año_encuesta  : año de la encuesta (necesario para transformaciones
                    que varían según el período histórico).
    """
    df  = df_in.copy()
    tr  = transformaciones

    # 1. NS/NR → NaN
    cols = [c for c in df.columns
            if c not in ("año", "pais_iso3", "pais_nombre", "ola")]
    df = limpiar_nsnr(df, cols, tr["nsnr"])

    # 2. Valores especiales
    if tr.get("a007071_nsnr_97") and "A_007_071" in df.columns:
        df.loc[df["A_007_071"] == 97, "A_007_071"] = np.nan
    if tr.get("c003003_nan5") and "C_003_003_011" in df.columns:
        df.loc[df["C_003_003_011"] == 5, "C_003_003_011"] = np.nan

    # 3. Likert 4 pts — confianza institucional
    for col in tr.get("likert4", []):
        if col not in df.columns:
            continue
        mask = df[col].notna() & df[col].between(1, 4)
        df.loc[mask, col] = 5 - df.loc[mask, col]

    # 4. Likert 4 pts — interés en política
    for col in tr.get("likert4_interes", []):
        if col not in df.columns:
            continue
        mask = df[col].notna() & df[col].between(1, 4)
        df.loc[mask, col] = 5 - df.loc[mask, col]

    # 5. Evaluaciones económicas comparativas
    for col in ["D_001_021", "D_001_041", "D_001_091"]:
        if col not in df.columns:
            continue
        if año_encuesta <= 2000:
            mask = df[col].between(1, 3)
            df.loc[mask, col] = 4 - df.loc[mask, col]
        else:
            mask = df[col].between(1, 5)
            df.loc[mask, col] = 6 - df.loc[mask, col]

    # 6. Recodificaciones binarias
    for col, mapeo in tr.get("binarias", {}).items():
        if col in df.columns:
            df[col] = df[col].map({float(k): v for k, v in mapeo.items()})

    # 7. Victimización — 3 períodos
    if "I_001_001" in df.columns:
        col   = "I_001_001"
        nueva = np.full(len(df), np.nan)
        if año_encuesta <= 2008:
            nueva[df[col].values == 1] = 1
            nueva[df[col].values == 2] = 0
        elif año_encuesta == 2009:
            nueva[np.isin(df[col].values, [1, 2])] = 1
            nueva[df[col].values == 3]              = 0
        else:
            nueva[np.isin(df[col].values, [1, 2, 3])] = 1
            nueva[df[col].values == 4]                 = 0
        df[col] = nueva

    # 8. Corrupción experiencial
    if "G_002_011" in df.columns:
        col = "G_002_011"
        if año_encuesta == 2013:
            df.loc[df[col] == 1, col] = 1
            df.loc[df[col] >  1, col] = 0
        else:
            df.loc[df[col] == 1, col] = 1
            df.loc[df[col] == 2, col] = 0

    # 9. V-Dem — invertir índices de corrupción
    for col in tr.get("vdem_invertir", []):
        if col in df.columns:
            df[col] = 1.0 - df[col]

    return df


# =============================================================================
# Función de predicción unificada para todos los modelos
# =============================================================================

def predecir(
    datos_crudos: dict,
    nombre_modelo: str = "XGBoost",
    subperiodo: str    = "SP3",
    año_encuesta: int  = 2024,
) -> dict:
    """
    Predice la satisfacción democrática a partir de valores originales
    del cuestionario de Latinobarómetro.

    Parámetros
    ----------
    datos_crudos  : dict con los valores originales sin transformar.
    nombre_modelo : 'OLO', 'XGBoost', 'CatBoost', 'LightGBM' o 'TabNet'.
    subperiodo    : 'SP1', 'SP2' o 'SP3'.
    año_encuesta  : año del cuestionario (determina transformaciones
                    que dependen del período histórico).

    Retorna
    -------
    dict con clase_predicha, etiqueta y probabilidades por clase.
    """
    ruta = CARPETA_MODELS / f"pipeline_{nombre_modelo}_{subperiodo}.pkl"
    assert ruta.exists(), f"Pipeline no encontrado: {ruta}"
    art  = joblib.load(ruta)
    tipo = art["tipo_modelo"]

    df_in = pd.DataFrame([datos_crudos])
    df_t  = aplicar_transformaciones_deterministas(
        df_in, art["transformaciones"], año_encuesta)
    df_t  = df_t.reindex(columns=art["features"])  # columnas faltantes → NaN

    feats = art["features"]

    if tipo == "olo":
        X_imp = pd.DataFrame(
            art["imp_num"].transform(df_t[feats]), columns=feats)
        X_sc  = pd.DataFrame(
            art["scaler"].transform(X_imp), columns=feats)
        y_pred = art["modelo"].predict(X_sc.values)
        y_prob = art["modelo"].predict_proba(X_sc.values)[0]

    elif tipo == "tabnet":
        X_imp = pd.DataFrame(
            art["imp_num"].transform(df_t[feats]), columns=feats)
        X_sc  = pd.DataFrame(
            art["scaler"].transform(X_imp), columns=feats)
        y_pred = art["modelo"].predict(X_sc.values.astype(np.float32))
        y_prob = art["modelo"].predict_proba(
                     X_sc.values.astype(np.float32))[0]

    else:  # trees: XGBoost, CatBoost, LightGBM
        X_in = df_t[feats].copy()
        if nombre_modelo == "CatBoost":
            for col in art.get("vars_categoricas", []):
                if col in X_in.columns:
                    X_in[col] = X_in[col].fillna(-999).astype(int).astype(str)
        y_raw  = art["modelo"].predict(X_in)
        y_pred = y_raw.flatten() if hasattr(y_raw, "flatten") else y_raw
        y_prob = art["modelo"].predict_proba(X_in)
        if y_prob.ndim == 2:
            y_prob = y_prob[0]

    clase = int(y_pred[0]) if hasattr(y_pred, "__len__") else int(y_pred)
    ets   = art["etiquetas_target"]
    return {
        "clase_predicha" : clase,
        "etiqueta"       : ets[clase],
        "probabilidades" : {ets[i]: float(p) for i, p in enumerate(y_prob)},
        "modelo"         : nombre_modelo,
        "subperiodo"     : subperiodo,
    }


print("✓ Funciones de producción definidas.")
print()
print("Ejemplo de uso:")
print("  resultado = predecir(")
print("      datos_crudos={'H_002_011': 3, 'D_001_001': 4, ...},")
print("      nombre_modelo='XGBoost',")
print("      subperiodo='SP3',")
print("      año_encuesta=2024,")
print("  )")

## 20. Tabla comparativa de resultados

In [ ]:
# =============================================================================
# Tabla comparativa de métricas por modelo y subperiodo
# =============================================================================
df_res = pd.DataFrame(RESULTADOS)
df_res = df_res.sort_values(["subperiodo","kappa_cuadratico"],
                             ascending=[True, False])

cols_show = ["modelo","subperiodo","split","balanced_accuracy","f1_macro",
             "kappa_lineal","kappa_cuadratico","mae_ordinal","auroc_macro"]
print("Métricas por modelo y subperiodo:")
print(df_res[[c for c in cols_show if c in df_res.columns]].to_string(
    index=False, float_format="{:.4f}".format))

print("\nKappa Cuadrático (modelo × subperiodo):")
pivot = df_res[df_res["split"]=="test"].pivot(index="modelo", columns="subperiodo",
                     values="kappa_cuadratico")
print(pivot.to_string(float_format="{:.4f}".format))

df_res.to_csv(CARPETA_RESULTS / "resultados_modelos.csv", index=False)
df_res.to_parquet(CARPETA_RESULTS / "resultados_modelos.parquet", index=False)
print("\n✓ Resultados guardados en results/")

## 21. Visualización comparativa de rendimiento

In [ ]:
# =============================================================================
# Gráficos de rendimiento por modelo y subperiodo
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    "Rendimiento comparativo de modelos\n"
    "(Expanding Window Walk-Forward Validation)",
    fontsize=13, fontweight="bold",
)

metricas_plot = [
    ("kappa_cuadratico", "Kappa Cuadrático (↑ mejor)"),
    ("f1_macro",         "F1 Macro (↑ mejor)"),
    ("mae_ordinal",      "MAE Ordinal (↓ mejor)"),
]
colores = {
    "OLO"    : "#2196F3", "XGBoost": "#4CAF50",
    "CatBoost":"#FF9800", "LightGBM":"#9C27B0", "TabNet":"#F44336",
}

for ax, (metrica, titulo) in zip(axes, metricas_plot):
    for modelo in ["OLO","XGBoost","CatBoost","LightGBM","TabNet"]:
        sub = df_res[df_res["modelo"] == modelo]
        if sub.empty or metrica not in sub.columns:
            continue
        ax.plot(sub["subperiodo"], sub[metrica],
                marker="o", linewidth=2, markersize=7,
                label=modelo, color=colores.get(modelo,"gray"))
    ax.set_title(titulo, fontweight="bold")
    ax.set_xlabel("Subperiodo")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    if metrica == "mae_ordinal":
        ax.invert_yaxis()

plt.tight_layout()
plt.savefig(CARPETA_RESULTS / "rendimiento_comparativo.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("✓ Figura guardada: rendimiento_comparativo.png")

## 22. Registro de versiones y entorno

In [ ]:
# =============================================================================
# Registro de versiones para reproducibilidad
# =============================================================================
import importlib, platform, sys

libs = ["pandas","numpy","sklearn","xgboost","lightgbm",
        "catboost","pytorch_tabnet","optuna","torch","missingno","joblib"]

print(f"Fecha          : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Python         : {sys.version.split()[0]}")
print(f"Sistema        : {platform.system()} {platform.release()}")
print(f"Semilla global : {SEED}")
print(f"GPU usada      : {USAR_GPU}")
print()
print("Versiones de librerías:")
for lib in libs:
    try:
        ver = getattr(importlib.import_module(lib), "__version__", "N/D")
        print(f"  {lib:<20}: {ver}")
    except ImportError:
        print(f"  {lib:<20}: NO INSTALADA")